# 7B QLoRA: Estonian invoice extraction
Select **Runtime → Change runtime type → T4 GPU**, then run top to bottom.
No prefilled training result, paid GPU or inference API. Free GPU availability is
controlled by Colab. Synthetic invoices measure a bounded extraction experiment,
not real-contract accuracy or expertise certification.

In [ ]:
import os,sys,subprocess,json
from pathlib import Path
CPU_CHECK=os.environ.get("QLORA_NOTEBOOK_CPU_CHECK")=="1"
ROOT=Path(os.environ["QLORA_NOTEBOOK_LOCAL_ROOT"]) if CPU_CHECK else Path("/content/llm-qlora-lab")
if not ROOT.exists():
    subprocess.run(["git","clone","https://github.com/mnoormets/llm-qlora-lab.git",str(ROOT)],check=True)
os.chdir(ROOT)
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
print("Source commit:",subprocess.check_output(["git","rev-parse","HEAD"],text=True).strip())
if not CPU_CHECK:subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements-gpu.txt"],check=True)
print("CPU contract check" if CPU_CHECK else "GPU experiment environment")

## Data and checks
400 training / 40 validation / 40 final-test examples. Disjoint entity IDs and held-out
wording templates. Missing fields use null. Do not tune against the final test.

In [ ]:
from qlora.data import load_rows
print((ROOT/"fixtures/manifest.json").read_text())
sets={name:load_rows(ROOT/"fixtures"/(name+".jsonl")) for name in ["train","validation","test"]}
assert not {r["id"] for r in sets["train"]}&{r["id"] for r in sets["test"]}
print("Validation preview:",json.dumps(sets["validation"][0],ensure_ascii=False))
subprocess.run([sys.executable,"-m","pytest","tests/test_data.py","-q"],check=True)

## GPU preflight and training
Uses fp16 on T4, bf16 only with native hardware support (emulation excluded). Default: 60 optimizer steps, batch 1,
accumulation 8, sequence limit 768. Compare baseline and adapter on the same
16 final-test cases. OOM produces a failure receipt, not a claimed result.
Do not start this on a paid runtime without agreeing its budget.

In [ ]:
if CPU_CHECK:
    print("GPU cells skipped during CPU validation; no 7B training result.")
else:
    import torch
    from qlora.train import preflight
    print(json.dumps(preflight(torch),indent=2))
    from qlora.runtime import run_logged
    run_logged([sys.executable,"-u","-m","qlora.train","--steps","60","--eval-cases","16"],ROOT/"runs"/"colab-training.log")

## Actual results
Read only completed receipts. JSON validity alone is not extraction accuracy.
Inspect exact match, field accuracy and every failed prediction.

In [ ]:
REPORT=None
reports=sorted((ROOT/"runs").glob("*/report.json"))
if reports and not CPU_CHECK:
    REPORT=reports[-1];result=json.loads(REPORT.read_text())
    assert result["status"]=="completed","Training did not complete"
    summary={k:result[k] for k in ["model","revision","hardware","dtype","steps","trainable_parameters","peak_allocated_gpu_gib","peak_reserved_gpu_gib","elapsed_seconds"]}
    summary["methods"]={name:{k:v for k,v in result[name].items() if k not in ["details","generation_seconds"]} for name in ["baseline","adapted"]}
    print(json.dumps(summary,indent=2))
    for item in result["adapted"]["details"]:
        if not item["exact_match"]:print(json.dumps(item,ensure_ascii=False))
else:print("No completed 7B GPU experiment in this session.")

## Retain artifacts
Download the adapter, tokenizer and report before the temporary runtime disappears.
The ZIP contains experiment artifacts, not credentials.

In [ ]:
if REPORT is not None:
    import shutil
    archive=shutil.make_archive("/content/qlora-experiment","zip",root_dir=REPORT.parent)
    from google.colab import files
    files.download(archive)
else:print("No GPU artifacts to download.")

## Optional private Hugging Face upload
Disabled by default. Review results first, then choose your account/repository.
The write token is entered through getpass and never printed or committed.
The uploaded adapter repository is private; review it before making it public.

In [ ]:
PUBLISH_TO_HUB=False
if PUBLISH_TO_HUB and REPORT is not None:
    from getpass import getpass
    from huggingface_hub import HfApi
    repository=input("Hugging Face account/repository: ").strip()
    token=getpass("Hugging Face write token: ")
    hub=HfApi(token=token)
    hub.create_repo(repository,repo_type="model",private=True,exist_ok=True)
    hub.upload_folder(repo_id=repository,folder_path=str(REPORT.parent/"adapter"),repo_type="model")
    hub.upload_file(path_or_fileobj=str(REPORT),path_in_repo="experiment-report.json",repo_id=repository,repo_type="model")
    del token,hub
    print("Private adapter and report uploaded.")
else:print("Hub upload disabled.")